# 🏧 Détection d'Anomalies sur le Réseau GAB — Analyse Journalière (v12)
## Référence 2025 & Reporting Comparatif 2024 / 2025 / 2026

---

> **Destinataires :** Experts métier – Responsables réseau GAB
> **Objectif :** Construire une **référence de comportement normal sur 2025**, puis **comparer 2024 et 2026 à cette référence** pour suivre l'évolution du réseau dans le temps et détecter les changements ou anomalies récurrentes.
> **Source des données :** dataset Dataiku `retrait` (sortie de la requête `ficheidentité.sql`) — **aucune donnée synthétique ou fictive** n'est utilisée dans ce notebook.
> **Maille d'analyse :** 1 ligne = 1 GAB × 1 jour (`annee`, `mois`, `jours`)

---

## Pourquoi une seule année de référence, appliquée aux deux autres ?

Deux approches étaient possibles pour comparer les 3 années disponibles (2024, 2025, 2026) :

1. **Un pipeline indépendant par année** (clustering et détection ré-entraînés séparément pour chaque année), puis comparaison des résultats de synthèse a posteriori.
2. **Un seul modèle de référence** (entraîné sur 2025), appliqué **tel quel** à 2024 et 2026 pour les scorer sur une base commune.

Ce notebook retient la **seconde approche**, pour deux raisons :

- **Comparabilité réelle.** Un clustering ré-entraîné à chaque année n'a aucune garantie de produire les mêmes familles d'une année sur l'autre (le K-Means n'a pas de notion d'identité stable entre deux exécutions séparées) — on ne pourrait jamais dire qu'un GAB « a changé de famille » entre deux années, faute de référentiel commun.
- **Mesure d'une vraie dérive, pas d'un artefact.** Si le modèle de 2026 est ré-entraîné sur les données de 2026 elles-mêmes, il apprend automatiquement à considérer tout nouveau comportement comme normal — même s'il s'agit d'une dérive réelle. Un modèle figé sur 2025 garde un point de vue fixe (« voici ce qu'on appelait normal en 2025 ») et mesure objectivement l'écart de 2024 et 2026 par rapport à ce point fixe.

La contrepartie assumée : si le réseau change structurellement dans le temps, le modèle 2025 peut devenir moins pertinent pour évaluer une année éloignée — une section dédiée (18) vérifie explicitement cette plausibilité plutôt que de l'ignorer.

| Partie | Rôle | Données utilisées |
|---|---|---|
| **1. Référence 2025** | Nettoyage, exploration, feature engineering, clustering, détection d'anomalies | **2025 uniquement** — tout `.fit()` est exécuté exclusivement sur 2025 |
| **2. Reporting comparatif 2024 / 2025 / 2026** | Application du modèle **figé** de 2025 à 2024 et 2026, comparaison des indicateurs à 3 années | **2024 et 2026**, scorés avec les objets (scaler, clustering, Isolation Forest) entraînés en Partie 1 — **jamais ré-entraînés** |

## ⚠️ Alignement strict sur la structure réelle des données

Ce notebook n'utilise **que** les colonnes réellement produites par `ficheidentité.sql` (dataset Dataiku `retrait`) :

- **Pas de colonne date** : uniquement `annee`, `mois`, `jours` (jour du mois, 1-31) séparés.
- **Distinction COS / hors-COS** partout : `ret_nb` (total), `ret_nb_cos`, `ret_nb_horscos` ; `cap_nb` (hors-COS), `cap_cos_nb`, `taux_capture_cos_pct`.
- **Motifs de capture détaillés** (hors-COS uniquement).
- **16 catégories de réseaux réelles** : Franfinance, CB, Trionis, PPL, Mastercard, Cofinoga, Interne, Casino, Accord, Amex, Visa, COS, JCB, Postépargne, Carte Diners et Discovery, Carte CUP, Autres.
- **Z-scores et `flag_atypique` déjà calculés en SQL**, comparant chaque jour à la moyenne/écart-type du **même mois** (`annee`+`mois`) sur l'ensemble du réseau — ce signal est repris tel quel comme référence de production, et complété (pas remplacé) par des z-scores intra-GAB calculés dans ce notebook.

## 1. ⚙️ Environnement d'exécution — Prérequis Dataiku DSS

Ce notebook est conçu pour tourner sous **Dataiku DSS**, kernel **Python 3.6 (distrib "new")**. Versions cibles vérifiées en environnement de production Dataiku :

| Librairie | Version cible Dataiku | Remarque |
|---|---|---|
| Python | 3.6.x | Pas de f-strings avec `=` (3.8+), pas de `match` (3.10+) |
| pandas | 1.1.5 | API stable utilisée ici |
| numpy | ≥ 1.16 | API stable |
| scikit-learn | 0.21.1 | **Pas de `HDBSCAN`** (ajouté en 1.3) ; **pas de `sparse_output=`** (ajouté en 1.2, non utilisé) ; `IsolationForest`, `KMeans`, `AgglomerativeClustering`, `GaussianMixture`, `silhouette_score`, `calinski_harabasz_score`, `davies_bouldin_score` stables depuis 0.20-0.21 |
| matplotlib | ≥ 3.0 | API stable |
| seaborn | ≥ 0.9 | `sns.set_style` stable |
| plotly | ≥ 4.0 | `plotly.express` stable |

⚠️ **Développement local vs exécution finale.** Ce notebook est développé/testé en local par confort, mais son exécution de référence est **exclusivement Dataiku DSS**. La cellule ci-dessous affiche les versions actives : un écart avec les cibles Dataiku n'est **pas bloquant en local** (c'est attendu), mais aucune fonctionnalité récente incompatible avec les cibles Dataiku n'est utilisée dans ce notebook.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Vérification des versions actives vs versions cibles Dataiku DSS
# ══════════════════════════════════════════════════════════════════════════════
import platform

VERSIONS_CIBLES_DATAIKU = {'python': '3.6', 'pandas': '1.1.5', 'scikit-learn': '0.21.1'}

print('-' * 65)
print('  VERSIONS ACTIVES DANS CE KERNEL')
print('-' * 65)
print('  Python        : {}  (cible Dataiku : {}.x)'.format(
    platform.python_version(), VERSIONS_CIBLES_DATAIKU['python']))

import pandas as pd
print('  pandas        : {}  (cible Dataiku : {})'.format(
    pd.__version__, VERSIONS_CIBLES_DATAIKU['pandas']))

import sklearn
print('  scikit-learn  : {}  (cible Dataiku : {})'.format(
    sklearn.__version__, VERSIONS_CIBLES_DATAIKU['scikit-learn']))

import numpy as np
print('  numpy         : {}'.format(np.__version__))

import matplotlib
print('  matplotlib    : {}'.format(matplotlib.__version__))

import seaborn as sns
print('  seaborn       : {}'.format(sns.__version__))

import plotly
print('  plotly        : {}'.format(plotly.__version__))

python_majeur_mineur = '.'.join(platform.python_version().split('.')[:2])
if python_majeur_mineur != VERSIONS_CIBLES_DATAIKU['python']:
    print('\n[ATTENTION] Kernel Python {} detecte (cible Dataiku : {}). '
          'Verifiez ce notebook sur Dataiku DSS avant mise en production.'.format(
              python_majeur_mineur, VERSIONS_CIBLES_DATAIKU['python']))
if pd.__version__ != VERSIONS_CIBLES_DATAIKU['pandas']:
    print('[ATTENTION] pandas {} detecte (cible : {}).'.format(
        pd.__version__, VERSIONS_CIBLES_DATAIKU['pandas']))
if sklearn.__version__ != VERSIONS_CIBLES_DATAIKU['scikit-learn']:
    print('[ATTENTION] scikit-learn {} detecte (cible : {}).'.format(
        sklearn.__version__, VERSIONS_CIBLES_DATAIKU['scikit-learn']))

## 2. 🧭 Imports et Configuration Générale

In [ ]:
# ── Librairies ───────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import pairwise_distances_argmin_min
from sklearn.decomposition import PCA

# ── Configuration graphique ───────────────────────────────────────────────────
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = 'white'
sns.set_style('whitegrid')

COULEUR_NORMAL    = '#2E7D32'
COULEUR_RECURRENT = '#C62828'
COULEUR_2025      = '#2c5282'
COULEUR_2026      = '#a03d3d'

RNG_SEED = 42
np.random.seed(RNG_SEED)

# ── Paramètres métier de ce notebook ──────────────────────────────────────────
NOM_DATASET_DATAIKU = 'retrait'
ANNEE_REFERENCE = 2025
ANNEES_COMPARAISON = [2024, 2026]

print('Configuration chargee.')
print('Dataset source      :', NOM_DATASET_DATAIKU)
print('Annee reference     :', ANNEE_REFERENCE)
print('Annees de comparaison:', ANNEES_COMPARAISON)

## 3. 🏗️ Chargement des Données et Séparation 2025 / 2026

Le dataset source `retrait` (sortie de `ficheidentité.sql`) est chargé une seule fois, puis **immédiatement séparé** en deux DataFrames distincts. Aucune étape de nettoyage, de feature engineering ou de modélisation n'est effectuée avant cette séparation, pour garantir qu'aucune fuite d'information entre les deux années n'est possible.

### Colonnes attendues (structure exacte de `ficheidentité.sql`)

- **Clés** : `num_automate`, `annee`, `mois`, `jours`
- **Référentiel** : `code_entite_de_gestion`, `type_gab_e_i`, `code_postale_emplacement`, `longitude`, `latitude`
- **Retraits** : `ret_nb`, `ret_nb_jours_actifs`, `ret_montant_total`, `ret_montant_moyen`, `ret_montant_max`, `ret_montant_min`, `ret_montant_stddev`, `ret_nb_nuit`, `ret_nb_weekend`, `ret_nb_heure_pointe`, `ret_heure_moyenne`, `ret_pct_nuit`, `ret_pct_weekend`, `ret_nb_cos`, `ret_nb_horscos`
- **Captures** : `cap_nb`, `cap_nb_nuit`, `cap_nb_weekend`, `cap_heure_moyenne`, `taux_capture_pct`, `cap_cos_nb`, `taux_capture_cos_pct`, `cap_nb_oubli_ou_incident_lecture`, `cap_nb_code_confidentiel_depasse`, `cap_nb_carte_perdue`, `cap_nb_carte_volee`, `cap_nb_autre_motif`, `cap_nb_motif_inconnu`
- **Réseaux** (16 catégories × `nb_ope_reseau_*` + `montant_reseau_*`) : franfinance, cb, trionis, ppl, mastercard, cofinoga, interne, casino, accord, amex, visa, cos, jcb, postepargne, carte_diners_et_discovery, carte_cup, autres
- **Z-scores et flag (déjà calculés en SQL, référence mensuelle)** : `zscore_ret_nb`, `zscore_ret_montant_total`, `zscore_ret_montant_moyen`, `zscore_ret_nb_nuit`, `zscore_ret_pct_nuit`, `zscore_ret_nb_weekend`, `zscore_cap_nb`, `zscore_taux_capture_pct`, `flag_atypique`

⚠️ **Il n'existe aucune colonne date** dans ce dataset — uniquement `annee`/`mois`/`jours` séparés.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Chargement du dataset réel Dataiku
# ══════════════════════════════════════════════════════════════════════════════
import dataiku

df_brut = dataiku.Dataset(NOM_DATASET_DATAIKU).get_dataframe()

print('Dataset charge :', NOM_DATASET_DATAIKU)
print('{:,} lignes x {} colonnes'.format(df_brut.shape[0], df_brut.shape[1]))
print('Colonnes :', list(df_brut.columns))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Séparation stricte par année — AUCUN mélange en aval
# 2025 = référence (Partie 1). 2024 et 2026 = comparés au modèle figé (Partie 2).
# ══════════════════════════════════════════════════════════════════════════════
df_2025 = df_brut[df_brut['annee'] == ANNEE_REFERENCE].copy()
df_par_annee_comparaison = {
    annee: df_brut[df_brut['annee'] == annee].copy() for annee in ANNEES_COMPARAISON
}
df_2024 = df_par_annee_comparaison[2024]
df_2026 = df_par_annee_comparaison[2026]

print('Annee {} (reference)   : {:,} lignes | {} GAB'.format(
    ANNEE_REFERENCE, len(df_2025), df_2025['num_automate'].nunique()))
for annee, df_a in df_par_annee_comparaison.items():
    print('Annee {} (comparaison): {:,} lignes | {} GAB'.format(
        annee, len(df_a), df_a['num_automate'].nunique()))

if len(df_2025) == 0:
    raise ValueError(
        "Aucune ligne trouvee pour l'annee de reference {}. "
        "Verifiez le contenu du dataset '{}' et la colonne 'annee'.".format(
            ANNEE_REFERENCE, NOM_DATASET_DATAIKU))
if len(df_2025) < 1000:
    print('\n[ATTENTION] Volume de donnees 2025 tres faible ({} lignes) '
          '- les resultats statistiques (clustering, Isolation Forest) '
          'seront peu fiables.'.format(len(df_2025)))
for annee, df_a in df_par_annee_comparaison.items():
    if len(df_a) == 0:
        print('\n[ATTENTION] Aucune ligne trouvee pour {} : cette annee '
              'ne pourra pas etre comparee en Partie 2.'.format(annee))

## 4. 🧩 Fonctions Communes de Feature Engineering

Ces fonctions sont définies **une seule fois** et appliquées séparément sur 2025 (avec ajustement/`fit`) puis sur 2026 (avec seulement `transform`/`predict`, jamais de ré-ajustement) — voir Partie 2.

Le tri chronologique se fait uniquement sur `(annee, mois, jours)`, seules colonnes disponibles pour ordonner les observations dans le temps (pas de colonne date dans le dataset source).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Listes de colonnes réelles — alignées sur ficheidentité.sql
# ══════════════════════════════════════════════════════════════════════════════
NOMS_RESEAUX = [
    'franfinance', 'cb', 'trionis', 'ppl', 'mastercard', 'cofinoga', 'interne',
    'casino', 'accord', 'amex', 'visa', 'cos', 'jcb', 'postepargne',
    'carte_diners_et_discovery', 'carte_cup', 'autres',
]
RESEAUX_ETRANGERS = ['jcb', 'amex', 'carte_cup', 'carte_diners_et_discovery']

COLS_RESEAU_NB = ['nb_ope_reseau_' + n for n in NOMS_RESEAUX]
COLS_ETRANGER_NB = ['nb_ope_reseau_' + n for n in RESEAUX_ETRANGERS]

COLS_MOTIFS_CAPTURE = [
    'cap_nb_oubli_ou_incident_lecture', 'cap_nb_code_confidentiel_depasse',
    'cap_nb_carte_perdue', 'cap_nb_carte_volee', 'cap_nb_autre_motif', 'cap_nb_motif_inconnu',
]


def trier_chronologiquement(df):
    """Tri par GAB puis par ordre calendaire (annee, mois, jours) - pas de colonne date."""
    return df.sort_values(['num_automate', 'annee', 'mois', 'jours']).reset_index(drop=True)


def calculer_features_reseau(df):
    """Ajoute nb_ope_total_reseau, pct_reseaux_etrangers, concentration_reseau (HHI)."""
    df = df.copy()
    df['nb_ope_total_reseau'] = df[COLS_RESEAU_NB].sum(axis=1)
    df['pct_reseaux_etrangers'] = (
        df[COLS_ETRANGER_NB].sum(axis=1) / (df['nb_ope_total_reseau'] + 1) * 100
    )
    parts = df[COLS_RESEAU_NB].div(df['nb_ope_total_reseau'] + 1, axis=0)
    df['concentration_reseau'] = (parts ** 2).sum(axis=1)
    return df


def calculer_bloc_a_niveau(df):
    """Bloc A - features de niveau (profil comportemental), 1 ligne = 1 GAB.
    Volume de reference = ret_nb_horscos (aligne sur la logique du SQL, qui base
    ses propres z-scores sur ret_nb_horscos plutot que sur ret_nb global).

    Les colonnes source (ret_montant_moyen, taux_capture_pct, taux_capture_cos_pct,
    pct_reseaux_etrangers) peuvent contenir des NaN au niveau jour - le SQL renvoie
    NULL quand le denominateur d'un ratio est nul un jour donne (ex: aucun retrait
    hors-COS ce jour-la). On utilise donc 'mean' avec skipna=True (comportement
    par defaut de pandas), ce qui suffit tant qu'un GAB a AU MOINS un jour valide
    dans l'annee. Le filet de securite ci-dessous couvre le cas plus rare d'un GAB
    n'ayant AUCUN jour valide sur toute la periode (moyenne alors NaN sur 100% du
    groupe), qui ferait planter PCA/RobustScaler en aval."""
    niveau_gab = df.groupby('num_automate').agg(
        code_postale_emplacement    = ('code_postale_emplacement', 'first'),
        longitude                   = ('longitude', 'first'),
        latitude                    = ('latitude', 'first'),
        volume_moyen_jour           = ('ret_nb_horscos', 'mean'),
        montant_moyen               = ('ret_montant_moyen', 'mean'),
        pct_nuit_moyen              = ('ret_pct_nuit', 'mean'),
        pct_weekend_moyen           = ('ret_pct_weekend', 'mean'),
        taux_capture_moyen          = ('taux_capture_pct', 'mean'),
        taux_capture_cos_moyen      = ('taux_capture_cos_pct', 'mean'),
        pct_reseaux_etrangers_moyen = ('pct_reseaux_etrangers', 'mean'),
        concentration_reseau_moyen  = ('concentration_reseau', 'mean'),
    ).reset_index()

    cols_a_verifier = [
        'volume_moyen_jour', 'montant_moyen', 'pct_nuit_moyen', 'pct_weekend_moyen',
        'taux_capture_moyen', 'taux_capture_cos_moyen', 'pct_reseaux_etrangers_moyen',
        'concentration_reseau_moyen',
    ]
    nb_nan_avant = niveau_gab[cols_a_verifier].isnull().sum().sum()
    if nb_nan_avant > 0:
        gab_incomplets = niveau_gab.loc[
            niveau_gab[cols_a_verifier].isnull().any(axis=1), 'num_automate'
        ].tolist()
        print('[ATTENTION] {} GAB sans aucune valeur exploitable sur au moins une '
              'metrique de niveau (moyenne restee NaN) : {}'.format(
                  len(gab_incomplets), gab_incomplets[:10]))
        print('   -> imputation par la mediane du reseau pour ces GAB, '
              'a defaut de mieux (donnees insuffisantes pour ces automates).')
        for col in cols_a_verifier:
            mediane = niveau_gab[col].median()
            niveau_gab[col] = niveau_gab[col].fillna(mediane)

    return niveau_gab


def calculer_zscore_intra_gab(df, metriques, fenetre=8, decalage=1, min_periodes=3):
    """Bloc B.2 - z-scores intra-GAB : compare chaque jour a l'historique PROPRE
    du GAB (moyenne/ecart-type sur une fenetre glissante decalee), complementaire
    aux zscore_* deja calcules en SQL (qui comparent au reseau entier, par mois).
    Le dataframe doit deja etre trie chronologiquement (trier_chronologiquement).

    ATTENTION - le dataset source (table retrait) exclut via un HAVING les jours
    sans aucun retrait : un GAB n'a donc en moyenne qu'une douzaine de lignes
    actives par an (pas ~365). La fenetre glissante porte ici sur un NOMBRE DE
    LIGNES ACTIVES (comportement natif de pandas .rolling(n) sur un DataFrame
    trie), pas sur des jours calendaires - fenetre=8 signifie "les 8 dernieres
    observations actives de ce GAB", quel que soit l'ecart calendaire entre
    elles. Des valeurs plus elevees (ex: fenetre=35, min_periodes=14, calibrees
    pour un historique quasi quotidien) videraient systematiquement le dropna()
    en aval et feraient planter StandardScaler (0 echantillon)."""
    df = df.copy()
    g = df.groupby('num_automate', sort=False)
    for m in metriques:
        decale = g[m].shift(decalage)
        moy_roulante = (decale.groupby(df['num_automate'])
                         .rolling(fenetre, min_periods=min_periodes).mean()
                         .reset_index(level=0, drop=True))
        std_roulante = (decale.groupby(df['num_automate'])
                         .rolling(fenetre, min_periods=min_periodes).std()
                         .reset_index(level=0, drop=True))
        df['zscore_intra_' + m] = (df[m] - moy_roulante) / std_roulante.replace(0, np.nan)
    return df


def assigner_cluster_le_plus_proche(x_scaled, centroides):
    """Assigne chaque ligne au centroide (moyenne de cluster) le plus proche,
    en distance euclidienne dans l'espace standardise. Utilise pour appliquer un
    clustering fige (KMeans avec .predict natif, ou Agglomerative/GaussianMixture
    sans predict natif compatible) a de nouvelles donnees."""
    labels, _ = pairwise_distances_argmin_min(x_scaled, centroides)
    return labels


print('Fonctions communes definies : trier_chronologiquement, calculer_features_reseau, '
      'calculer_bloc_a_niveau, calculer_zscore_intra_gab, assigner_cluster_le_plus_proche.')

---
# PARTIE 1 — Année de Référence 2025

Toutes les cellules ci-dessous n'utilisent **exclusivement** `df_2025`. Aucun `.fit()`, aucune moyenne, aucun écart-type n'est calculé sur des données incluant 2026.
---

## 5. 🧹 Prétraitement et Audit Qualité — 2025

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Audit qualité — 2025
# ══════════════════════════════════════════════════════════════════════════════
print('-' * 65)
print('  AUDIT DE QUALITE DES DONNEES - 2025')
print('-' * 65)

manquants = df_2025.isnull().sum()
if manquants.sum() > 0:
    print('\n[ATTENTION] Valeurs manquantes : {}'.format(manquants[manquants > 0].to_dict()))
else:
    print('\nAucune valeur manquante detectee.')

doublons = df_2025.duplicated(subset=['num_automate', 'annee', 'mois', 'jours']).sum()
print('Doublons (num_automate, annee, mois, jours) : {}'.format(doublons))

incoherence_ret = (df_2025['ret_nb_cos'] + df_2025['ret_nb_horscos'] != df_2025['ret_nb']).sum()
print('Coherence ret_nb = ret_nb_cos + ret_nb_horscos : {} lignes incoherentes'.format(incoherence_ret))

somme_motifs = df_2025[COLS_MOTIFS_CAPTURE].sum(axis=1)
incoherence_motifs = (somme_motifs != df_2025['cap_nb']).sum()
print('Coherence motifs de capture = cap_nb : {} lignes incoherentes'.format(incoherence_motifs))

# NULL en base SQL quand un ratio/moyenne n'a aucune ligne pour le calculer
# (ex: taux_capture_pct est NULL un jour sans aucun retrait hors-COS ce jour-la).
# On comble ces NaN par 0 avant tout calcul de moyenne par GAB, sinon ils se
# propagent silencieusement jusqu'a PCA/RobustScaler et font planter le notebook.
cols_a_combler_zero = [
    'ret_montant_total', 'ret_montant_moyen', 'ret_montant_max', 'ret_montant_min',
    'ret_montant_stddev', 'ret_pct_nuit', 'ret_pct_weekend', 'taux_capture_pct',
    'taux_capture_cos_pct',
]
nb_nan_avant_comblement = df_2025[cols_a_combler_zero].isnull().sum()
if nb_nan_avant_comblement.sum() > 0:
    print('[ATTENTION] Valeurs NULL comblees par 0 (aucun retrait/capture ce jour-la) : {}'.format(
        nb_nan_avant_comblement[nb_nan_avant_comblement > 0].to_dict()))
df_2025[cols_a_combler_zero] = df_2025[cols_a_combler_zero].fillna(0)

for col in ['ret_pct_nuit', 'ret_pct_weekend']:
    df_2025[col] = df_2025[col].clip(0, 100)
for col in ['taux_capture_pct', 'taux_capture_cos_pct']:
    df_2025[col] = df_2025[col].clip(lower=0)
df_2025['ret_nb'] = df_2025['ret_nb'].clip(0)
df_2025['cap_nb'] = df_2025['cap_nb'].clip(0)

cols_reseau_all = COLS_RESEAU_NB + ['montant_reseau_' + n for n in NOMS_RESEAUX]
df_2025[cols_reseau_all] = df_2025[cols_reseau_all].fillna(0).clip(lower=0)

print('\nDataset 2025 pret : {:,} lignes x {} colonnes'.format(df_2025.shape[0], df_2025.shape[1]))
print('GAB : {}'.format(df_2025['num_automate'].nunique()))

## 6. 🧬 Feature Engineering — 2025

Deux blocs disjoints :

| Bloc | Contenu | Usage | Échelle |
|---|---|---|---|
| **A — Niveau** | Profil comportemental moyen sur 2025 | Clustering (familles) | 1 ligne = 1 GAB |
| **B — Variabilité** | Z-scores SQL (mensuels, déjà calculés) + z-scores intra-GAB (nouveaux, complémentaires) | Détection d'anomalie | 1 ligne = 1 GAB × 1 jour |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOC A — Features de niveau, 2025
# ══════════════════════════════════════════════════════════════════════════════
df_2025 = trier_chronologiquement(df_2025)
df_2025 = calculer_features_reseau(df_2025)

niveau_gab_2025 = calculer_bloc_a_niveau(df_2025)

print('Bloc A (niveau) 2025 : {} GAB x {} features'.format(*niveau_gab_2025.shape))
niveau_gab_2025.describe().T[['mean', 'std', 'min', 'max']].round(2)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOC B — Z-scores intra-GAB (complément aux zscore_* déjà calculés en SQL), 2025
# ══════════════════════════════════════════════════════════════════════════════
METRIQUES_INTRA = ['ret_nb_horscos', 'ret_montant_moyen', 'taux_capture_pct', 'ret_pct_nuit']

df_2025 = calculer_zscore_intra_gab(df_2025, METRIQUES_INTRA)

# ATTENTION : 'zscore_intra_...' commence aussi par 'zscore_' - on exclut
# explicitement les colonnes intra du filtre SQL pour ne pas les compter deux fois.
cols_zscore_intra = [c for c in df_2025.columns if c.startswith('zscore_intra_')]
cols_zscore_sql = [c for c in df_2025.columns
                   if c.startswith('zscore_') and c not in cols_zscore_intra]

print('Z-scores SQL (mensuels, deja calcules en production) :', cols_zscore_sql)
print('Z-scores intra-GAB (nouveaux, complementaires) :', cols_zscore_intra)
print('\nLignes sans z-score intra (periode d\'amorcage, historique insuffisant) : {:,}'.format(
    df_2025['zscore_intra_ret_nb_horscos'].isnull().sum()))

## 7. 🔬 Diagnostic Anti-Colinéarité — 2025

Avant tout clustering, on vérifie que les features du **bloc A** ne sont pas redondantes. Pour toute paire avec |corrélation de Pearson| > 0.85, une seule feature est retenue.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Matrice de corrélation — features de niveau 2025
# ══════════════════════════════════════════════════════════════════════════════
FEATS_NIVEAU_CANDIDATES = [
    'volume_moyen_jour', 'montant_moyen', 'pct_nuit_moyen', 'pct_weekend_moyen',
    'taux_capture_moyen', 'taux_capture_cos_moyen', 'pct_reseaux_etrangers_moyen',
    'concentration_reseau_moyen',
]

corr_2025 = niveau_gab_2025[FEATS_NIVEAU_CANDIDATES].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_2025, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Corrélation entre features de niveau (bloc A) — 2025', fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_features_niveau_2025.png', dpi=150, bbox_inches='tight')
plt.show()

mask_triu = np.triu(np.ones(corr_2025.shape), k=1).astype(bool)
paires_corr = corr_2025.where(mask_triu).stack().sort_values(key=abs, ascending=False)
paires_fortes = paires_corr[paires_corr.abs() > 0.85]

print('Paires de features avec |r| > 0.85 :')
for (f1, f2), r in paires_fortes.items():
    print('   {:30s} <-> {:30s} : r = {:+.3f}'.format(f1, f2, r))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Élimination des redondances — jeu final de features de clustering (2025)
# ══════════════════════════════════════════════════════════════════════════════
# Cette liste sera reprise TELLE QUELLE en Partie 2 pour scorer 2026 -
# elle ne doit jamais etre recalculee sur 2026.
FEATS_A_EXCLURE = []
paires_a_traiter = paires_fortes.copy()
for (f1, f2), r in paires_a_traiter.items():
    if f1 not in FEATS_A_EXCLURE and f2 not in FEATS_A_EXCLURE:
        FEATS_A_EXCLURE.append(f2)

FEATS_NIVEAU_CLUSTERING = [f for f in FEATS_NIVEAU_CANDIDATES if f not in FEATS_A_EXCLURE]

print('Features retenues pour le clustering ({}) :'.format(len(FEATS_NIVEAU_CLUSTERING)))
for f in FEATS_NIVEAU_CLUSTERING:
    print('   - {}'.format(f))
print('\nFeatures exclues (redondantes) : {}'.format(FEATS_A_EXCLURE))

X_diag_2025 = RobustScaler().fit_transform(niveau_gab_2025[FEATS_NIVEAU_CLUSTERING])
pca_diag = PCA(random_state=RNG_SEED).fit(X_diag_2025)
var_cum = np.cumsum(pca_diag.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(range(1, len(var_cum) + 1), pca_diag.explained_variance_ratio_, alpha=0.6, label='Variance par composante')
ax.plot(range(1, len(var_cum) + 1), var_cum, 'o-', color=COULEUR_RECURRENT, label='Variance cumulée')
ax.axhline(0.8, color='grey', linestyle='--', linewidth=1, label='Seuil 80%')
ax.set_xlabel('Composante principale'); ax.set_ylabel('Variance expliquée')
ax.set_title('PCA — Variance expliquée (2025)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('pca_variance_expliquee_2025.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nLes {} premieres composantes expliquent {:.1f}% de la variance totale.'.format(
    2, var_cum[1] * 100))

## 8. 📊 Analyse Exploratoire Journalière — 2025

In [ ]:
# ── Saisonnalité — vue d'ensemble du réseau, 2025 ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

vol_mois = df_2025.groupby('mois')['ret_nb_horscos'].mean()
axes[0].plot(vol_mois.index, vol_mois.values, 'o-', color=COULEUR_2025, linewidth=2)
axes[0].set_xticks(range(1, 13))
axes[0].set_title('Volume moyen de retraits (hors-COS) par mois — 2025', fontweight='bold')
axes[0].set_xlabel('Mois'); axes[0].set_ylabel('Retraits hors-COS / jour (moyenne réseau)')

taux_capture_mois = df_2025.groupby('mois')['taux_capture_pct'].mean()
axes[1].plot(taux_capture_mois.index, taux_capture_mois.values, 'o-', color=COULEUR_RECURRENT, linewidth=2)
axes[1].set_xticks(range(1, 13))
axes[1].set_title('Taux de capture moyen par mois — 2025', fontweight='bold')
axes[1].set_xlabel('Mois'); axes[1].set_ylabel('Taux de capture (%)')

plt.tight_layout()
plt.savefig('saisonnalite_2025.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. 🤖 Détection des Journées Atypiques — Isolation Forest — 2025

Le modèle est entraîné sur les **z-scores intra-GAB** (bloc B, complémentaires aux z-scores SQL déjà présents). Les objets `scaler_anom_2025` et `iso_forest_2025` sont conservés pour être appliqués tels quels à 2026 en Partie 2 (jamais ré-entraînés).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Isolation Forest — entraîné exclusivement sur 2025
# ══════════════════════════════════════════════════════════════════════════════
FEATS_ANOMALIE = cols_zscore_sql + cols_zscore_intra

df_model_2025 = df_2025.dropna(subset=cols_zscore_intra).copy()

X_anom_2025 = df_model_2025[FEATS_ANOMALIE].fillna(0)
scaler_anom_2025 = StandardScaler()
X_anom_2025_scaled = scaler_anom_2025.fit_transform(X_anom_2025)

CONTAMINATION = 0.01

iso_forest_2025 = IsolationForest(n_estimators=200, contamination=CONTAMINATION,
                                   random_state=RNG_SEED, n_jobs=-1)
df_model_2025['score_anomalie_jour'] = iso_forest_2025.fit_predict(X_anom_2025_scaled)
df_model_2025['score_if_continu']    = iso_forest_2025.score_samples(X_anom_2025_scaled)
df_model_2025['est_atypique_jour']   = (df_model_2025['score_anomalie_jour'] == -1).astype(int)

n_atyp = df_model_2025['est_atypique_jour'].sum()
print('Isolation Forest entraîné sur {:,} jours-GAB (2025), {} features.'.format(
    X_anom_2025_scaled.shape[0], X_anom_2025_scaled.shape[1]))
print('Jours-GAB détectés atypiques : {:,} ({:.2f}%)'.format(n_atyp, n_atyp / len(df_model_2025) * 100))

### ✅ Note méthodologique — validation sans vérité terrain

Aucune donnée de ce notebook n'étant simulée, il n'existe **aucun label d'anomalie connu** pour valider ce modèle par précision/rappel. La validation repose sur :

1. **Plausibilité statistique** — comparaison des GAB détectés aux z-scores extrêmes (déjà en partie couverte par `flag_atypique`, calculé en SQL).
2. **Stabilité par ré-échantillonnage** — à réaliser en aval de ce notebook si besoin (ré-entraîner sur des sous-échantillons de GAB, vérifier la stabilité des détections).
3. **Revue métier a posteriori** — faire valider un échantillon de détections par les responsables réseau.
4. **Cohérence inter-signaux** — recouvrement entre `flag_atypique` (SQL, mensuel) et `est_atypique_jour` (ce notebook, journalier).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cohérence entre le signal SQL (flag_atypique, mensuel) et Isolation Forest (journalier)
# ══════════════════════════════════════════════════════════════════════════════
recouvrement = pd.crosstab(df_model_2025['flag_atypique'], df_model_2025['est_atypique_jour'])
print('Recouvrement flag_atypique (SQL) x est_atypique_jour (Isolation Forest) :')
print(recouvrement)

## 10. 📆 Agrégation Multi-Échelle — 2025

Sans colonne date, l'agrégation hebdomadaire n'est pas calculable nativement (pas de semaine ISO). L'agrégation se fait donc au niveau **mois**, cohérent avec le `flag_atypique` du SQL (déjà calculé par mois) : on identifie les GAB en **dérive récurrente** (plusieurs mois consécutifs avec au moins un jour atypique) par opposition à un signal ponctuel ou au bruit de fond.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Agrégation jour → mois, détection de séries de mois consécutifs atypiques
# ══════════════════════════════════════════════════════════════════════════════
agg_mois_2025 = df_model_2025.groupby(['num_automate', 'mois']).agg(
    nb_jours_atypiques = ('est_atypique_jour', 'sum'),
).reset_index()
agg_mois_2025['mois_atypique'] = (agg_mois_2025['nb_jours_atypiques'] > 0).astype(int)

tous_les_mois = pd.DataFrame({'mois': sorted(df_model_2025['mois'].unique())})
idx_complet = pd.MultiIndex.from_product(
    [df_model_2025['num_automate'].unique(), tous_les_mois['mois']],
    names=['num_automate', 'mois']
)
agg_mois_full_2025 = (agg_mois_2025.set_index(['num_automate', 'mois'])['mois_atypique']
                       .reindex(idx_complet, fill_value=0).reset_index()
                       .sort_values(['num_automate', 'mois']))

est_stable = (agg_mois_full_2025['mois_atypique'] == 0)
groupe_streak = est_stable.groupby(agg_mois_full_2025['num_automate']).cumsum()
streak_courant = agg_mois_full_2025.groupby(['num_automate', groupe_streak]).cumcount() + 1
agg_mois_full_2025['streak_courant'] = streak_courant.where(agg_mois_full_2025['mois_atypique'] == 1, 0)

streak_max_gab_2025 = agg_mois_full_2025.groupby('num_automate')['streak_courant'].max()

SEUIL_STREAK_RECURRENT = 3
classification_gab_2025 = pd.Series(
    np.where(streak_max_gab_2025 >= SEUIL_STREAK_RECURRENT,
             'DERIVE RECURRENTE DETECTEE', 'Pas de derive durable'),
    index=streak_max_gab_2025.index, name='statut_derive'
)

print('Classification par GAB (streak de mois consécutifs atypiques) — 2025 :')
print(classification_gab_2025.value_counts())

## 11. 🔬 Diagnostic de Séparabilité — 2025

Pour chaque `k` de 2 à 10 et 3 algorithmes (`KMeans`, `AgglomerativeClustering`, `GaussianMixture` — tous disponibles en scikit-learn 0.21.1), on calcule silhouette, Calinski-Harabasz et Davies-Bouldin. Le couple (k, algorithme) retenu est celui qui maximise objectivement ces métriques — jamais fixé a priori.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Grille comparative : features de niveau × k=2..10 × 3 algorithmes — 2025
# ══════════════════════════════════════════════════════════════════════════════
X_cluster_2025 = RobustScaler().fit_transform(niveau_gab_2025[FEATS_NIVEAU_CLUSTERING])

resultats_diag = []
for k in range(2, 11):
    candidats = {
        'KMeans':          KMeans(n_clusters=k, n_init=10, random_state=RNG_SEED),
        'Agglomerative':    AgglomerativeClustering(n_clusters=k, linkage='ward'),
        'GaussianMixture': GaussianMixture(n_components=k, random_state=RNG_SEED, n_init=3),
    }
    for algo_nom, algo in candidats.items():
        labels = algo.fit_predict(X_cluster_2025)
        if len(set(labels)) < 2:
            continue
        resultats_diag.append({
            'k': k, 'algorithme': algo_nom,
            'silhouette':        silhouette_score(X_cluster_2025, labels),
            'calinski_harabasz': calinski_harabasz_score(X_cluster_2025, labels),
            'davies_bouldin':    davies_bouldin_score(X_cluster_2025, labels),
        })

diag_df_2025 = pd.DataFrame(resultats_diag)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, metric, titre in [
    (axes[0], 'silhouette', 'Silhouette (plus haut = mieux)'),
    (axes[1], 'calinski_harabasz', 'Calinski-Harabasz (plus haut = mieux)'),
    (axes[2], 'davies_bouldin', 'Davies-Bouldin (plus bas = mieux)'),
]:
    for algo_nom in diag_df_2025['algorithme'].unique():
        sub = diag_df_2025[diag_df_2025['algorithme'] == algo_nom]
        ax.plot(sub['k'], sub[metric], 'o-', label=algo_nom)
    ax.set_xlabel('k (nombre de clusters)'); ax.set_title(titre, fontweight='bold')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('diagnostic_separabilite_2025.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 combinaisons (k, algorithme) par silhouette :')
print(diag_df_2025.sort_values('silhouette', ascending=False).head(5).to_string(index=False))

## 12. 🧩 Clustering Final des Familles Comportementales — 2025

Le meilleur score de silhouette pur donne souvent `k=2` de façon dégénérée (un cluster capte l'essentiel des GAB). On retient le plus petit `k ≥ 3` produisant des clusters d'une taille minimale exploitable (aucun cluster < 3% des GAB).

**Ce modèle (scaler + algorithme + centroïdes) est figé ici et réutilisé tel quel en Partie 2 pour scorer 2026 — jamais ré-entraîné.**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Sélection objective du (k, algorithme) — 2025
# ══════════════════════════════════════════════════════════════════════════════
TAILLE_MIN_PCT = 0.03

def taille_min_ok(k, algo_nom):
    candidats = {
        'KMeans':          KMeans(n_clusters=k, n_init=10, random_state=RNG_SEED),
        'Agglomerative':    AgglomerativeClustering(n_clusters=k, linkage='ward'),
        'GaussianMixture': GaussianMixture(n_components=k, random_state=RNG_SEED, n_init=3),
    }
    labels = candidats[algo_nom].fit_predict(X_cluster_2025)
    tailles = pd.Series(labels).value_counts(normalize=True)
    return tailles.min() >= TAILLE_MIN_PCT, labels

silhouette_max = diag_df_2025['silhouette'].max()
diag_valides = diag_df_2025[(diag_df_2025['silhouette'] >= silhouette_max - 0.02) & (diag_df_2025['k'] >= 3)].copy()
diag_valides = diag_valides.sort_values('k')

K_FINAL, ALGO_FINAL, LABELS_FINAL_2025 = None, None, None
for _, row in diag_valides.iterrows():
    ok, labels = taille_min_ok(int(row['k']), row['algorithme'])
    if ok:
        K_FINAL, ALGO_FINAL, LABELS_FINAL_2025 = int(row['k']), row['algorithme'], labels
        break

if K_FINAL is None:
    row = diag_df_2025[diag_df_2025['k'] >= 3].sort_values('silhouette', ascending=False).iloc[0]
    K_FINAL, ALGO_FINAL = int(row['k']), row['algorithme']
    _, LABELS_FINAL_2025 = taille_min_ok(K_FINAL, ALGO_FINAL)

niveau_gab_2025['cluster'] = LABELS_FINAL_2025
print('Choix retenu : k={}, algorithme={}'.format(K_FINAL, ALGO_FINAL))
print('Répartition : {}'.format(pd.Series(LABELS_FINAL_2025).value_counts().sort_index().to_dict()))

# ── Objets figés, réutilisés en Partie 2 ──────────────────────────────────────
scaler_cluster_2025 = RobustScaler().fit(niveau_gab_2025[FEATS_NIVEAU_CLUSTERING])
X_scaled_2025 = scaler_cluster_2025.transform(niveau_gab_2025[FEATS_NIVEAU_CLUSTERING])

centroides_liste = []
for cid in sorted(niveau_gab_2025['cluster'].unique()):
    masque = (niveau_gab_2025['cluster'] == cid).values
    centroides_liste.append(X_scaled_2025[masque].mean(axis=0))
CENTROIDES_2025 = np.vstack(centroides_liste)

print('Centroïdes 2025 figés (forme {}) - à réutiliser tels quels en Partie 2.'.format(CENTROIDES_2025.shape))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Profil des familles et nommage — par traits dominants réels, 2025
# ══════════════════════════════════════════════════════════════════════════════
profil_clusters_2025 = niveau_gab_2025.groupby('cluster')[FEATS_NIVEAU_CLUSTERING].mean()
moyenne_globale_2025 = niveau_gab_2025[FEATS_NIVEAU_CLUSTERING].mean()
ecart_relatif_2025 = (profil_clusters_2025 - moyenne_globale_2025) / moyenne_globale_2025

NOMS_COURTS_FEATS = {
    'volume_moyen_jour': 'volume', 'montant_moyen': 'montant',
    'pct_nuit_moyen': 'nocturne', 'pct_weekend_moyen': 'weekend',
    'taux_capture_moyen': 'capture', 'taux_capture_cos_moyen': 'capture COS',
    'pct_reseaux_etrangers_moyen': 'réseaux étrangers', 'concentration_reseau_moyen': 'concentration réseau',
}

def nommer_cluster(cid, ecarts_df):
    ecarts = ecarts_df.loc[cid].sort_values(key=abs, ascending=False)
    trait_principal = ecarts.index[0]
    signe = 'plus' if ecarts.iloc[0] > 0 else 'moins'
    return 'Famille {} ({} {})'.format(cid, signe, NOMS_COURTS_FEATS.get(trait_principal, trait_principal))

NOMS_CLUSTERS_2025 = {cid: nommer_cluster(cid, ecart_relatif_2025) for cid in profil_clusters_2025.index}
niveau_gab_2025['cluster_nom'] = niveau_gab_2025['cluster'].map(NOMS_CLUSTERS_2025)

print('Profil moyen par famille (écart relatif à la moyenne du réseau) — 2025 :')
display(profil_clusters_2025.rename(index=NOMS_CLUSTERS_2025).round(2))

print('\nRépartition :')
print(niveau_gab_2025['cluster_nom'].value_counts())

## 13. 🗺️ Carte Géographique — Familles vs Statut de Dérive — 2025

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Carte double — familles comportementales vs statut de dérive, 2025
# ══════════════════════════════════════════════════════════════════════════════
niveau_gab_2025['statut_derive'] = niveau_gab_2025['num_automate'].map(classification_gab_2025)

PALETTE_FAMILLES = ['#2c5282', '#a03d3d', '#2f6b4f', '#8a6d3b', '#6b4c8a', '#3d7a99']
noms_familles_2025 = sorted(niveau_gab_2025['cluster_nom'].unique())
couleur_famille_2025 = {nom: PALETTE_FAMILLES[i % len(PALETTE_FAMILLES)] for i, nom in enumerate(noms_familles_2025)}
couleur_statut = {'Pas de derive durable': COULEUR_NORMAL, 'DERIVE RECURRENTE DETECTEE': COULEUR_RECURRENT}

fig_2025 = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'mapbox'}, {'type': 'mapbox'}]],
    subplot_titles=('Familles comportementales — 2025', 'Statut de dérive détecté — 2025'),
    horizontal_spacing=0.03,
)

for nom in noms_familles_2025:
    sub = niveau_gab_2025[niveau_gab_2025['cluster_nom'] == nom]
    fig_2025.add_trace(go.Scattermapbox(
        lat=sub['latitude'], lon=sub['longitude'], mode='markers',
        marker=dict(size=10, color=couleur_famille_2025[nom]),
        text=sub['num_automate'], hovertemplate='%{text}<br>' + nom + '<extra></extra>',
        name=nom, legendgroup='familles',
    ), row=1, col=1)

for statut, couleur in couleur_statut.items():
    sub = niveau_gab_2025[niveau_gab_2025['statut_derive'] == statut]
    fig_2025.add_trace(go.Scattermapbox(
        lat=sub['latitude'], lon=sub['longitude'], mode='markers',
        marker=dict(size=10, color=couleur),
        text=sub['num_automate'], hovertemplate='%{text}<br>' + statut + '<extra></extra>',
        name=statut, legendgroup='statut',
    ), row=1, col=2)

fig_2025.update_layout(
    mapbox=dict(style='carto-positron', zoom=4.2, center=dict(lat=46.6, lon=2.2)),
    mapbox2=dict(style='carto-positron', zoom=4.2, center=dict(lat=46.6, lon=2.2)),
    height=520, margin=dict(r=10, t=50, l=10, b=10),
    legend=dict(orientation='h', yanchor='top', y=-0.05),
)
fig_2025.show()

print('Répartition par famille x statut de dérive — 2025 :')
print(pd.crosstab(niveau_gab_2025['cluster_nom'], niveau_gab_2025['statut_derive']))

## 14. 🔎 Tableau de Bord de Synthèse — 2025

C'est la **photographie de référence** : 1 ligne = 1 GAB, avec sa famille comportementale et son statut de dérive sur 2025.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Synthèse consolidée par GAB — 2025
# ══════════════════════════════════════════════════════════════════════════════
intensite_anomalie_2025 = df_model_2025.groupby('num_automate').agg(
    nb_jours_observes  = ('est_atypique_jour', 'count'),
    nb_jours_atypiques = ('est_atypique_jour', 'sum'),
    score_if_moyen     = ('score_if_continu', 'mean'),
).reset_index()
intensite_anomalie_2025['pct_jours_atypiques'] = (
    intensite_anomalie_2025['nb_jours_atypiques'] / intensite_anomalie_2025['nb_jours_observes'] * 100
).round(2)

tableau_bord_2025 = niveau_gab_2025[['num_automate', 'code_postale_emplacement', 'cluster_nom', 'statut_derive']].merge(
    intensite_anomalie_2025[['num_automate', 'nb_jours_atypiques', 'pct_jours_atypiques']],
    on='num_automate'
).merge(
    streak_max_gab_2025.rename('streak_mois_max'), on='num_automate'
).sort_values(['statut_derive', 'pct_jours_atypiques'], ascending=[True, False])

print('Tableau de bord 2025 — {} GAB'.format(len(tableau_bord_2025)))
display(tableau_bord_2025.head(15))

print('\nGAB à investiguer en priorité (dérive récurrente détectée) : {}'.format(
    (tableau_bord_2025['statut_derive'] == 'DERIVE RECURRENTE DETECTEE').sum()))

---
# PARTIE 2 — Reporting Comparatif 2024 / 2025 / 2026

Toutes les cellules ci-dessous appliquent à 2024 et 2026 les objets **entraînés exclusivement sur 2025** en Partie 1 (`scaler_cluster_2025`, `CENTROIDES_2025`, `NOMS_CLUSTERS_2025`, `scaler_anom_2025`, `iso_forest_2025`, `FEATS_NIVEAU_CLUSTERING`, `FEATS_ANOMALIE`, `METRIQUES_INTRA`). **Aucun `.fit()` n'est appelé sur 2024 ni sur 2026.**

Une seule fonction (`traiter_annee_comparee`) est définie et appliquée successivement aux deux années, pour éviter de dupliquer la logique.
---

## 16. 🧹 Prétraitement, Feature Engineering et Scoring — Fonction Commune

Pour chaque année comparée (2024, 2026), la même séquence est appliquée : audit qualité, feature engineering (blocs A et B avec les mêmes fonctions communes qu'en Partie 1), puis scoring avec les objets figés de 2025 (`transform`/`predict`, jamais de `fit`).

Pour les z-scores intra-GAB, la fenêtre glissante est autorisée à chevaucher les **derniers jours de l'année précédente** (ex. fin 2023 pour amorcer 2024, fin 2025 pour amorcer 2026) — uniquement pour éviter une période d'amorçage vide en tout début d'année. Le buffer n'est jamais lui-même analysé ni réinjecté dans un clustering.

⚠️ Le buffer d'amorçage pour **2024** nécessiterait des données de fin **2023**, non disponibles dans ce notebook (années disponibles : 2024, 2025, 2026). Dans ce cas, les tout premiers jours de janvier 2024 auront une période d'amorçage incomplète (comme pour janvier 2025 en Partie 1) — c'est documenté, pas masqué.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Fonction générique : traite une année comparée avec les objets figés de 2025
# ══════════════════════════════════════════════════════════════════════════════
def traiter_annee_comparee(df_annee, annee, df_annee_precedente=None):
    """Applique le pipeline complet (audit, feature engineering, scoring) à une
    année de comparaison, en utilisant EXCLUSIVEMENT les objets figés sur 2025
    (scaler_cluster_2025, CENTROIDES_2025, NOMS_CLUSTERS_2025, scaler_anom_2025,
    iso_forest_2025). Aucun .fit() n'est appelé ici.

    df_annee_precedente : DataFrame de l'année N-1, utilisé uniquement pour
    amorcer la fenêtre glissante des z-scores intra-GAB (jamais analysé lui-même).
    """
    print('-' * 65)
    print('  TRAITEMENT DE L\'ANNEE {}'.format(annee))
    print('-' * 65)

    if len(df_annee) == 0:
        print('[ATTENTION] Aucune donnee pour {} - annee ignoree.'.format(annee))
        return None

    df_a = df_annee.copy()

    # Audit qualité minimal (mêmes règles qu'en Partie 1)
    incoherence_ret = (df_a['ret_nb_cos'] + df_a['ret_nb_horscos'] != df_a['ret_nb']).sum()
    print('Coherence ret_nb = ret_nb_cos + ret_nb_horscos : {} lignes incoherentes'.format(incoherence_ret))
    # NULL en base SQL quand un ratio/moyenne n'a aucune ligne pour le calculer -
    # meme comblement que pour 2025 (voir cellule d'audit qualite 2025).
    cols_a_combler_zero = [
        'ret_montant_total', 'ret_montant_moyen', 'ret_montant_max', 'ret_montant_min',
        'ret_montant_stddev', 'ret_pct_nuit', 'ret_pct_weekend', 'taux_capture_pct',
        'taux_capture_cos_pct',
    ]
    df_a[cols_a_combler_zero] = df_a[cols_a_combler_zero].fillna(0)

    for col in ['ret_pct_nuit', 'ret_pct_weekend']:
        df_a[col] = df_a[col].clip(0, 100)
    for col in ['taux_capture_pct', 'taux_capture_cos_pct']:
        df_a[col] = df_a[col].clip(lower=0)
    df_a['ret_nb'] = df_a['ret_nb'].clip(0)
    df_a['cap_nb'] = df_a['cap_nb'].clip(0)
    cols_reseau_all = COLS_RESEAU_NB + ['montant_reseau_' + n for n in NOMS_RESEAUX]
    df_a[cols_reseau_all] = df_a[cols_reseau_all].fillna(0).clip(lower=0)

    # Bloc A - features de niveau
    df_a = trier_chronologiquement(df_a)
    df_a = calculer_features_reseau(df_a)
    niveau_gab_a = calculer_bloc_a_niveau(df_a)
    print('Bloc A (niveau) {} : {} GAB x {} features'.format(annee, *niveau_gab_a.shape))

    # Bloc B - z-scores intra-GAB, buffer optionnel de l'année précédente
    if df_annee_precedente is not None and len(df_annee_precedente) > 0:
        buffer_annee_prec = (df_annee_precedente.sort_values(['num_automate', 'mois', 'jours'])
                              .groupby('num_automate').tail(N_JOURS_BUFFER))
        df_a_avec_buffer = pd.concat([buffer_annee_prec, df_a], ignore_index=True)
        df_a_avec_buffer = trier_chronologiquement(df_a_avec_buffer)
        df_a_avec_buffer = calculer_zscore_intra_gab(df_a_avec_buffer, METRIQUES_INTRA)
        df_a = df_a_avec_buffer[df_a_avec_buffer['annee'] == annee].copy()
        print('Z-scores intra-GAB {} calcules avec buffer de fin d\'annee precedente.'.format(annee))
    else:
        df_a = calculer_zscore_intra_gab(df_a, METRIQUES_INTRA)
        print('[ATTENTION] Pas de buffer d\'annee precedente pour {} - periode '
              'd\'amorcage en debut d\'annee incomplete (attendu, documente).'.format(annee))

    # Application du clustering figé 2025 (transform + assignation au centroïde le plus proche)
    X_cluster_a = scaler_cluster_2025.transform(niveau_gab_a[FEATS_NIVEAU_CLUSTERING])
    niveau_gab_a['cluster'] = assigner_cluster_le_plus_proche(X_cluster_a, CENTROIDES_2025)
    niveau_gab_a['cluster_nom'] = niveau_gab_a['cluster'].map(NOMS_CLUSTERS_2025)

    # Application de l'Isolation Forest figé 2025 (transform + predict, jamais fit)
    df_model_a = df_a.dropna(subset=[c for c in df_a.columns if c.startswith('zscore_intra_')]).copy()
    X_anom_a = df_model_a[FEATS_ANOMALIE].fillna(0)
    X_anom_a_scaled = scaler_anom_2025.transform(X_anom_a)
    df_model_a['est_atypique_jour'] = (iso_forest_2025.predict(X_anom_a_scaled) == -1).astype(int)
    df_model_a['score_if_continu'] = iso_forest_2025.score_samples(X_anom_a_scaled)

    n_atyp = df_model_a['est_atypique_jour'].sum()
    print('Jours-GAB detectes atypiques ({}) : {:,} ({:.2f}%)'.format(
        annee, n_atyp, n_atyp / len(df_model_a) * 100 if len(df_model_a) > 0 else 0))

    return {
        'annee': annee,
        'df_prepare': df_a,
        'df_model': df_model_a,
        'niveau_gab': niveau_gab_a,
        'X_cluster': X_cluster_a,
    }


print('Fonction traiter_annee_comparee definie.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Application aux deux années de comparaison (2024 et 2026) — AUCUN .fit() ici
# ══════════════════════════════════════════════════════════════════════════════
resultats_2024 = traiter_annee_comparee(df_2024, 2024, df_annee_precedente=None)
resultats_2026 = traiter_annee_comparee(df_2026, 2026, df_annee_precedente=df_2025)

resultats_par_annee = {}
if resultats_2024 is not None:
    resultats_par_annee[2024] = resultats_2024
resultats_par_annee[2025] = {
    'annee': 2025, 'df_prepare': df_2025, 'df_model': df_model_2025,
    'niveau_gab': niveau_gab_2025, 'X_cluster': X_cluster_2025,
}
if resultats_2026 is not None:
    resultats_par_annee[2026] = resultats_2026

print('\nAnnees disponibles pour le reporting comparatif : {}'.format(sorted(resultats_par_annee.keys())))

## 18. ⚠️ Vérification de Plausibilité — Le Modèle 2025 Reste-t-il Pertinent ?

Un modèle figé peut devenir obsolète si la structure du réseau change fondamentalement dans le temps. On compare la distance moyenne des GAB de chaque année comparée à leur centroïde assigné, à la distance moyenne intra-cluster observée en 2025 elle-même — un écart important signale que le modèle 2025 représente mal cette année.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Distance aux centroïdes — référence (2025) vs chaque année comparée
# ══════════════════════════════════════════════════════════════════════════════
_, distances_2025 = pairwise_distances_argmin_min(X_cluster_2025, CENTROIDES_2025)
dist_moyenne_2025 = distances_2025.mean()

print('Distance moyenne au centroïde assigné :')
print('   2025 (référence) : {:.3f}'.format(dist_moyenne_2025))

for annee in sorted(resultats_par_annee.keys()):
    if annee == 2025:
        continue
    res = resultats_par_annee[annee]
    _, distances_a = pairwise_distances_argmin_min(res['X_cluster'], CENTROIDES_2025)
    dist_moyenne_a = distances_a.mean()
    ecart_pct = (dist_moyenne_a - dist_moyenne_2025) / dist_moyenne_2025 * 100
    print('   {} : {:.3f}  (ecart vs reference : {:+.1f}%)'.format(annee, dist_moyenne_a, ecart_pct))
    if ecart_pct > 30:
        print('      [ATTENTION] Les GAB {} sont sensiblement plus eloignes de leurs '
              'centroides 2025 - le modele de reference pourrait ne plus bien '
              'representer cette annee.'.format(annee))

## 19. 🤖 Détection des Journées Atypiques — Récapitulatif 2024 / 2025 / 2026

Rappel : la détection pour 2024 et 2026 utilise l'Isolation Forest **figé sur 2025** (`iso_forest_2025.predict(...)`), jamais ré-entraîné.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Taux de jours atypiques par année (base commune : modèle figé 2025)
# ══════════════════════════════════════════════════════════════════════════════
recap_atypique = []
for annee in sorted(resultats_par_annee.keys()):
    df_m = resultats_par_annee[annee]['df_model']
    n_atyp = df_m['est_atypique_jour'].sum()
    recap_atypique.append({
        'annee': annee,
        'nb_jours_observes': len(df_m),
        'nb_jours_atypiques': int(n_atyp),
        'pct_jours_atypiques': round(n_atyp / len(df_m) * 100, 2) if len(df_m) > 0 else 0,
    })

recap_atypique_df = pd.DataFrame(recap_atypique)
print('Taux de jours atypiques par année (modèle figé 2025) :')
display(recap_atypique_df)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(recap_atypique_df['annee'].astype(str), recap_atypique_df['pct_jours_atypiques'],
       color=[COULEUR_2025 if a == 2025 else COULEUR_2026 for a in recap_atypique_df['annee']])
ax.set_ylabel('% de jours-GAB atypiques')
ax.set_title('Taux de jours atypiques par année — modèle figé 2025', fontweight='bold')
plt.tight_layout()
plt.savefig('taux_atypique_par_annee.png', dpi=150, bbox_inches='tight')
plt.show()

## 20. 📊 Comparatif des Indicateurs Clés — 2024 / 2025 / 2026

Tableau côte-à-côte des indicateurs de niveau (volume, montant, % nocturne, taux de capture, % réseaux étrangers), avec écarts en % par rapport à la référence 2025.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Comparatif des indicateurs de niveau — 3 années
# ══════════════════════════════════════════════════════════════════════════════
indicateurs_par_annee = {}
for annee in sorted(resultats_par_annee.keys()):
    niv = resultats_par_annee[annee]['niveau_gab']
    indicateurs_par_annee[annee] = niv[FEATS_NIVEAU_CLUSTERING].mean()

comparatif_indicateurs = pd.DataFrame(indicateurs_par_annee).round(2)
comparatif_indicateurs.columns = ['{}'.format(a) for a in comparatif_indicateurs.columns]

for annee in comparatif_indicateurs.columns:
    if annee != str(ANNEE_REFERENCE):
        comparatif_indicateurs['ecart_{}_vs_{}(%)'.format(annee, ANNEE_REFERENCE)] = (
            (comparatif_indicateurs[annee] - comparatif_indicateurs[str(ANNEE_REFERENCE)])
            / comparatif_indicateurs[str(ANNEE_REFERENCE)] * 100
        ).round(1)

print('Comparatif des indicateurs clés (moyenne réseau) — 2024 / 2025 / 2026 :')
display(comparatif_indicateurs)

## 21. 🔀 Évolution des Familles Comportementales par GAB

Un GAB peut-il changer de famille assignée d'une année à l'autre (2025 → 2024, 2025 → 2026), toujours par rapport aux **mêmes familles définies en 2025** ? Table de transition par GAB.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Table de transition des familles par GAB, entre 2025 et chaque année comparée
# ══════════════════════════════════════════════════════════════════════════════
for annee in sorted(resultats_par_annee.keys()):
    if annee == ANNEE_REFERENCE:
        continue
    niv_a = resultats_par_annee[annee]['niveau_gab']
    fusion = niveau_gab_2025[['num_automate', 'cluster_nom']].merge(
        niv_a[['num_automate', 'cluster_nom']], on='num_automate',
        suffixes=('_2025', '_{}'.format(annee))
    )
    print('Transition des familles : 2025 -> {}'.format(annee))
    print(pd.crosstab(fusion['cluster_nom_2025'], fusion['cluster_nom_{}'.format(annee)]))
    print()

## 22. 🔎 Anomalies Récurrentes et Nouvelles Dérives

GAB en dérive récurrente sur **plusieurs années** (signal persistant, prioritaire) vs GAB nouvellement en dérive **seulement en 2026** (n'étaient pas en dérive en 2025) vs GAB dont la dérive s'est **résorbée** entre 2025 et 2026.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Statut de dérive par année (mois consécutifs atypiques, seuil = 3, comme 2025)
# ══════════════════════════════════════════════════════════════════════════════
SEUIL_STREAK_RECURRENT_COMPARAISON = 3

def calculer_statut_derive(df_model_annee):
    agg_mois = df_model_annee.groupby(['num_automate', 'mois']).agg(
        nb_jours_atypiques=('est_atypique_jour', 'sum')
    ).reset_index()
    agg_mois['mois_atypique'] = (agg_mois['nb_jours_atypiques'] > 0).astype(int)

    tous_mois = pd.DataFrame({'mois': sorted(df_model_annee['mois'].unique())})
    idx = pd.MultiIndex.from_product(
        [df_model_annee['num_automate'].unique(), tous_mois['mois']],
        names=['num_automate', 'mois']
    )
    agg_full = (agg_mois.set_index(['num_automate', 'mois'])['mois_atypique']
                .reindex(idx, fill_value=0).reset_index()
                .sort_values(['num_automate', 'mois']))

    est_stable = (agg_full['mois_atypique'] == 0)
    groupe = est_stable.groupby(agg_full['num_automate']).cumsum()
    streak = agg_full.groupby(['num_automate', groupe]).cumcount() + 1
    agg_full['streak'] = streak.where(agg_full['mois_atypique'] == 1, 0)
    streak_max = agg_full.groupby('num_automate')['streak'].max()

    return pd.Series(
        np.where(streak_max >= SEUIL_STREAK_RECURRENT_COMPARAISON,
                 'DERIVE RECURRENTE DETECTEE', 'Pas de derive durable'),
        index=streak_max.index, name='statut_derive'
    )


statuts_par_annee = {ANNEE_REFERENCE: classification_gab_2025}
for annee in sorted(resultats_par_annee.keys()):
    if annee == ANNEE_REFERENCE:
        continue
    statuts_par_annee[annee] = calculer_statut_derive(resultats_par_annee[annee]['df_model'])

for annee, statut in statuts_par_annee.items():
    print('Statut de derive {} :'.format(annee))
    print(statut.value_counts())
    print()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Recoupement : GAB en dérive récurrente sur plusieurs années
# ══════════════════════════════════════════════════════════════════════════════
if 2026 in statuts_par_annee and ANNEE_REFERENCE in statuts_par_annee:
    derive_2025 = set(statuts_par_annee[ANNEE_REFERENCE][
        statuts_par_annee[ANNEE_REFERENCE] == 'DERIVE RECURRENTE DETECTEE'].index)
    derive_2026 = set(statuts_par_annee[2026][
        statuts_par_annee[2026] == 'DERIVE RECURRENTE DETECTEE'].index)

    derive_persistante = derive_2025 & derive_2026
    derive_nouvelle_2026 = derive_2026 - derive_2025
    derive_resorbee = derive_2025 - derive_2026

    print('GAB en dérive persistante (2025 ET 2026) : {}'.format(len(derive_persistante)))
    print('   {}'.format(sorted(derive_persistante)[:20]))
    print('\nGAB en NOUVELLE dérive (2026 uniquement, pas en 2025) : {}'.format(len(derive_nouvelle_2026)))
    print('   {}'.format(sorted(derive_nouvelle_2026)[:20]))
    print('\nGAB en dérive RESORBEE (2025 uniquement, plus en 2026) : {}'.format(len(derive_resorbee)))
    print('   {}'.format(sorted(derive_resorbee)[:20]))

## 23. 📋 Tableau de Bord Comparatif Final

1 ligne = 1 GAB, avec son statut de dérive pour chaque année disponible et une colonne de synthèse de l'évolution.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Tableau de bord comparatif — 1 ligne = 1 GAB, toutes années
# ══════════════════════════════════════════════════════════════════════════════
tableau_comparatif = niveau_gab_2025[['num_automate', 'code_postale_emplacement']].copy()

for annee, statut in statuts_par_annee.items():
    statut_df = statut.rename('statut_derive_{}'.format(annee)).reset_index()
    statut_df.columns = ['num_automate', 'statut_derive_{}'.format(annee)]
    tableau_comparatif = tableau_comparatif.merge(statut_df, on='num_automate', how='left')

def synthese_evolution(row):
    s25 = row.get('statut_derive_{}'.format(ANNEE_REFERENCE))
    s26 = row.get('statut_derive_2026')
    if s25 is None or s26 is None:
        return 'Non evaluable'
    derive_25 = (s25 == 'DERIVE RECURRENTE DETECTEE')
    derive_26 = (s26 == 'DERIVE RECURRENTE DETECTEE')
    if derive_25 and derive_26:
        return 'Derive persistante'
    if not derive_25 and derive_26:
        return 'Nouvelle derive (2026)'
    if derive_25 and not derive_26:
        return 'Derive resorbee'
    return 'Stable'

tableau_comparatif['evolution'] = tableau_comparatif.apply(synthese_evolution, axis=1)

print('Tableau de bord comparatif — {} GAB'.format(len(tableau_comparatif)))
display(tableau_comparatif.head(15))

print('\nRépartition des évolutions :')
print(tableau_comparatif['evolution'].value_counts())

## 24. 🗺️ Carte Géographique — Évolution 2025 → 2026

Visualisation géographique des GAB selon leur trajectoire d'évolution (stable / nouvelle dérive / dérive résorbée / dérive persistante).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Carte de l'évolution des GAB entre 2025 et 2026
# ══════════════════════════════════════════════════════════════════════════════
carte_evolution = niveau_gab_2025[['num_automate', 'latitude', 'longitude']].merge(
    tableau_comparatif[['num_automate', 'evolution']], on='num_automate'
)

couleur_evolution = {
    'Stable': COULEUR_NORMAL,
    'Nouvelle derive (2026)': COULEUR_RECURRENT,
    'Derive persistante': '#6b1616',
    'Derive resorbee': '#8a6d3b',
    'Non evaluable': '#999999',
}

fig_evolution = px.scatter_mapbox(
    carte_evolution, lat='latitude', lon='longitude',
    color='evolution',
    color_discrete_map=couleur_evolution,
    hover_name='num_automate',
    mapbox_style='carto-positron',
    zoom=4.2, center={'lat': 46.6, 'lon': 2.2},
    title='Évolution des GAB — 2025 → 2026',
    height=560,
)
fig_evolution.update_layout(margin=dict(r=10, t=50, l=10, b=10))
fig_evolution.show()

---
## 25. 📌 Conclusion Générale & Méthodologie de Validation

### Ce que ce notebook apporte par rapport aux versions précédentes

| Avant | Ce notebook (v12) |
|---|---|
| Toutes les années mélangées dans un seul pipeline | Séparation stricte : 2025 = référence figée, 2024/2026 = comparés |
| Dataset synthétique/fictif | Chargement réel exclusif du dataset Dataiku `retrait` |
| Colonnes inventées (`date_iso`, `is_ferie`, etc.) | Colonnes strictement alignées sur `ficheidentité.sql` |
| Un seul niveau de comparaison | Z-scores SQL (mensuels, référence réseau) + z-scores intra-GAB (nouveaux, référence propre au GAB) |
| Clustering re-entraîné à chaque analyse | Clustering et Isolation Forest **figés sur 2025**, appliqués tels quels à 2024 et 2026 |

### Pourquoi un modèle figé plutôt que ré-entraîné chaque année

Un modèle ré-entraîné sur les données de l'année qu'il évalue absorbe automatiquement toute dérive comme un nouveau "normal" — il ne peut alors plus détecter cette dérive. Un modèle figé sur 2025 conserve un point de comparaison fixe, ce qui permet de répondre à la vraie question métier : *"ce comportement diverge-t-il de ce qu'on considérait comme normal en 2025 ?"*

### ⚠️ Limites assumées de ce notebook

- **Le modèle 2025 peut devenir moins pertinent avec le temps** si la structure du réseau change fondamentalement (nouveaux types de GAB, nouvelle réglementation) — la section 18 (vérification de plausibilité) surveille cet écart, sans le masquer.
- **Absence de vérité terrain** — aucune donnée de ce notebook n'étant simulée, il n'existe aucun label d'anomalie connu pour mesurer précision/rappel. La validation repose sur la plausibilité statistique, la cohérence avec `flag_atypique` (SQL), et une revue métier a posteriori à mettre en place.
- **Buffer d'amorçage incomplet pour 2024** — faute de données de fin 2023, les tout premiers jours de janvier 2024 ont une période d'amorçage incomplète pour les z-scores intra-GAB (documenté en section 16).
- **Séparabilité du clustering** — le nombre de familles comportementales dépend entièrement du signal réellement présent dans les données 2025 ; ce notebook ne force jamais un nombre de familles a priori.

### Méthodologie de validation transposable, sans vérité terrain

1. **Plausibilité statistique** — les GAB détectés sont-ils vraiment aux extrêmes des distributions (cohérence avec `flag_atypique` calculé en SQL, section 9) ?
2. **Stabilité par ré-échantillonnage** — ré-entraîner le clustering/Isolation Forest sur des sous-échantillons de GAB 2025, vérifier que les mêmes GAB ressortent de façon stable.
3. **Revue métier a posteriori** — faire valider un échantillon de GAB détectés (dérive persistante, nouvelle dérive) par les responsables réseau, qui disposent d'un historique d'incidents réels.
4. **Cohérence inter-signaux** — recouper `est_atypique_jour` (Isolation Forest), `flag_atypique` (SQL) et un éventuel historique de tickets de maintenance, si disponible.

### Prochaines étapes suggérées

- Faire valider le tableau de bord comparatif (section 23) par les responsables réseau, en priorité sur les GAB en **dérive persistante** et en **nouvelle dérive 2026**.
- Envisager un ré-entraînement périodique de la référence (par exemple tous les 2 ans) si la section 18 signale un écart de plausibilité croissant.
- Enrichir le bloc A avec des variables métier supplémentaires si elles deviennent disponibles (historique de maintenance, type de matériel GAB).